In [0]:
%sql
-- create volume
CREATE EXTERNAL VOLUME dev.bronze.landing
    LOCATION "abfss://devlake@storagelakedevsample.dfs.core.windows.net/volumes"
    COMMENT "landing zone"

In [0]:
# create autoloader_input folder in Volumn
dbutils.fs.mkdirs("/Volumes/dev/bronze/landing/autoloader_input/2010/12/01")
dbutils.fs.mkdirs("/Volumes/dev/bronze/landing/autoloader_input/2010/12/02")
dbutils.fs.mkdirs("/Volumes/dev/bronze/landing/autoloader_input/2010/12/03")
dbutils.fs.mkdirs("/Volumes/dev/bronze/landing/autoloader_input/2010/12/04")
dbutils.fs.mkdirs("/Volumes/dev/bronze/landing/autoloader_input/2010/12/05")
dbutils.fs.mkdirs("/Volumes/dev/bronze/landing/autoloader_input/2010/12/06")
dbutils.fs.mkdirs("/Volumes/dev/bronze/landing/autoloader_input/2010/12/07")

In [0]:
# create checkpoint
dbutils.fs.mkdirs("/Volumes/dev/bronze/landing/checkpoint/autoloader")

In [0]:

# copy file to nested location
# dbutils.fs.ls("/databricks-datasets/definitive-guide/data/retail-data/by-day/")
dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-01.csv", "/Volumes/dev/bronze/landing/autoloader_input/2010/12/01")
dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-02.csv", "/Volumes/dev/bronze/landing/autoloader_input/2010/12/02")
dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-03.csv", "/Volumes/dev/bronze/landing/autoloader_input/2010/12/03")

In [0]:
# read file using autoloader with checkpoint
# File detection modes:
## directory listing
## File notification

df = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "csv")
        .option("pathGlobFilter", "*.csv")
        .option("header", "true")
        .option("cloudFiles.schemaHints", "Quantity int, UnitPrice double")
        .option("cloudFiles.schemaLocation", "/Volumes/dev/bronze/landing/checkpoint/autoloader/1/")
        .load("/Volumes/dev/bronze/landing/autoloader_input/*/")
    )


In [0]:
# write data to delta table
from pyspark.sql.functions import col
(
    df.withColumn("__file", col("_metadata.file_path"))
        .writeStream 
        .format("delta") 
        .option("checkpointLocation", "/Volumes/dev/bronze/landing/checkpoint/autoloader/1/")
        .option("mergeSchema", True)
        .outputMode("Append")
        .trigger(availableNow=True)
        .toTable("dev.bronze.invoice_al_1")
        
)

In [0]:
%sql

select __file, count(1)
from dev.bronze.invoice_al_1
group by __file

In [0]:
%sql

select *
from dev.bronze.invoice_al_1
where notes is not null

In [0]:

dbutils.fs.cp("/databricks-datasets/definitive-guide/data/retail-data/by-day/2010-12-05.csv", "/Volumes/dev/bronze/landing/autoloader_input/2010/12/05")

In [0]:
# re-run read and write cell above